In [0]:
from pyspark.sql import functions as F, types as T

data = [
    ("2017-01-01", 32.0,  6.0,  "Rain"),
    ("2017-01-04", None,  9.0,  "Sunny"),
    ("2017-01-05", 28.0,  None, "Snow"),
    ("2017-01-06", None,  7.0,  None),
    ("2017-01-07", 32.0,  None, "Rain"),
    ("2017-01-08", None,  None, "Sunny"),
    ("2017-01-09", None,  None, None),
    ("2017-01-10", 34.1,  8.1,  "Cloudy"),
    ("2017-01-11", 40.0, 12.0,  "Sunny"),
]

schema = "day string, temperature double, windspeed double, event string"
df = spark.createDataFrame(data, schema)
df = df.withColumn("day", F.to_date("day", "yyyy-MM-dd"))  # normalize to DateType

display(df)

In [0]:
df.createOrReplaceTempView("weather")

In [0]:
%sql
Select event, Round(avg(temperature),1) as Temp
From weather
Group By event
Order By Temp desc;

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1,  "Asha",  "IN", True),
    (2,  "Bob",   "US", False),
    (3,  "Chen",  "CN", True),
    (4,  "Diana", "US", None),
    (None, "Ghost","UK", False),     # NULL key to demo null join behavior
]

rows_orders = [
    (101, 1,   120.0, "IN"),
    (102, 1,    80.0, "IN"),
    (103, 2,    50.0, "US"),
    (104, 5,    30.0, "DE"),         # no matching customer_id
    (105, 3,   200.0, "CN"),
    (106, None, 15.0, "UK"),         # NULL key won’t match
    (107, 3,    40.0, "CN"),
    (108, 2,    75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name",        T.StringType(),  True),
    T.StructField("country",     T.StringType(),  True),
    T.StructField("vip",         T.BooleanType(), True),
])

schema_orders = T.StructType([
    T.StructField("order_id",    T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount",      T.DoubleType(),  True),
    T.StructField("country",     T.StringType(),  True),  # same column name to show collisions
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders    = spark.createDataFrame(rows_orders,    schema_orders)

display(df_customers)
display(df_orders)

In [0]:
df_inner=df_orders.join(df_customers,on="customer_id",how="inner")
display(df_inner)

✔ DataFrame basics

read CSV

select columns

filter rows

add derived columns

✔ SQL practice

register temp view

write SELECT queries

simple JOIN (orders ↔ customers)

Read CSV

In [0]:
df_n=spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/workspace/default/raw_data_1/orders.csv")

In [0]:
display(df_n)

Select columns

In [0]:
df_col=df_n.select("order_id","order_date","product","price")
display(df_col)

Filter Rows


In [0]:
df_n_filtered=df_n.filter((df_n.product=="Shoes") & (df_n.price<60))
display(df_n_filtered)